In [24]:
# import os
# import shutil
# from sklearn.model_selection import train_test_split

# images_dir = "data3folders/images"
# labels_dir = "data3folders/labels"

# # Collect all images
# images = [f for f in os.listdir(images_dir) if f.endswith(".jpg") or f.endswith(".png")]

# # Split 70/20/10
# train_val_images, test_images = train_test_split(images, test_size=0.1, random_state=42)
# train_images, val_images = train_test_split(train_val_images, test_size=0.222, random_state=42)

# # Create folders
# for split in ["train", "val", "test"]:
#     os.makedirs(f"{images_dir}/{split}", exist_ok=True)
#     os.makedirs(f"{labels_dir}/{split}", exist_ok=True)

# # Move into splits
# for img in train_images:
#     name = os.path.splitext(img)[0]
#     shutil.move(f"{images_dir}/{img}", f"{images_dir}/train/{img}")
#     shutil.move(f"{labels_dir}/{name}.txt", f"{labels_dir}/train/{name}.txt")

# for img in val_images:
#     name = os.path.splitext(img)[0]
#     shutil.move(f"{images_dir}/{img}", f"{images_dir}/val/{img}")
#     shutil.move(f"{labels_dir}/{name}.txt", f"{labels_dir}/val/{name}.txt")

# for img in test_images:
#     name = os.path.splitext(img)[0]
#     shutil.move(f"{images_dir}/{img}", f"{images_dir}/test/{img}")
#     shutil.move(f"{labels_dir}/{name}.txt", f"{labels_dir}/test/{name}.txt")

# print(f"Train: {len(train_images)} images")
# print(f"Val:   {len(val_images)} images")
# print(f"Test:  {len(test_images)} images")

In [25]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="datafolders.yaml",
    epochs=20,
    imgsz=512,
)

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.46  Python-3.13.7 torch-2.11.0+cpu CPU (AMD Ryzen 5 9600X 6-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datafolders.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-1

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000025126C51550>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [26]:
from ultralytics import YOLO
from PIL import Image, ImageDraw
import cv2
import numpy as np
import torchvision.ops as ops
import torch

def tile_and_detect(image_path, model, conf=0.45):
    img_cv = cv2.imread(image_path)
    img_shape = img_cv.shape  # (H, W, C)

    # Match training tiling logic exactly
    count, rem = divmod(img_shape[0], 512)
    overlap = (1 - (rem / 512)) * 512 / count
    vertical_offset = int(512 - overlap)

    count, rem = divmod(img_shape[1], 512)
    overlap = (1 - (rem / 512)) * 512 / count
    horizontal_offset = int(512 - overlap)

    column_range = int(img_shape[0] / vertical_offset)
    results_all = []

    for j in range(column_range):
        row_range = int(img_shape[1] / horizontal_offset)
        for k in range(row_range):
            y = vertical_offset * j
            x = horizontal_offset * k
            tile = img_cv[y:min(y + 512, img_shape[0]), x:min(x + 512, img_shape[1])]

            results = model(tile, conf=conf, imgsz=512)
            for result in results:
                for box in result.boxes:
                    bx1, by1, bx2, by2 = box.xyxy[0].tolist()
                    results_all.append((
                        x + bx1, y + by1, x + bx2, y + by2,
                        box.conf[0].item()
                    ))

    if not results_all:
        print("No detections found.")
        cv2.imwrite("detection_output.jpg", img_cv)
        return []

    # NMS to remove duplicates from overlapping tiles
    boxes = torch.tensor([[x1, y1, x2, y2] for x1, y1, x2, y2, _ in results_all])
    scores = torch.tensor([conf for *_, conf in results_all])
    keep = ops.nms(boxes, scores, iou_threshold=0.3)
    results_all = [results_all[i] for i in keep]

    # Draw crosshairs
    for (x1, y1, x2, y2, conf) in results_all:
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
        cv2.circle(img_cv, (cx, cy), 40, (0, 0, 255), 6)
        cv2.line(img_cv, (cx - 40, cy), (cx + 40, cy), (0, 0, 255), 4)
        cv2.line(img_cv, (cx, cy - 40), (cx, cy + 40), (0, 0, 255), 4)
        cv2.putText(img_cv, f"{conf:.2f}", (cx - 40, cy - 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)

    annotated = img_cv.copy()

    cv2.imshow("Detections", annotated)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    return results_all

model = YOLO("runs/detect/train/weights/best.pt")
detections = tile_and_detect("data/images/val/65d57ec7-DJI_20260323164055_0193_D.jpg", model)
for det in detections:
    print(f"GCP found at {det[:4]} with confidence {det[4]:.2f}")


0: 512x512 (no detections), 16.1ms
Speed: 0.9ms preprocess, 16.1ms inference, 0.3ms postprocess per image at shape (1, 3, 512, 512)

0: 512x512 (no detections), 23.2ms
Speed: 0.5ms preprocess, 23.2ms inference, 0.4ms postprocess per image at shape (1, 3, 512, 512)

0: 512x512 (no detections), 15.7ms
Speed: 0.7ms preprocess, 15.7ms inference, 0.3ms postprocess per image at shape (1, 3, 512, 512)

0: 512x512 (no detections), 16.0ms
Speed: 0.7ms preprocess, 16.0ms inference, 0.4ms postprocess per image at shape (1, 3, 512, 512)

0: 512x512 (no detections), 17.4ms
Speed: 0.6ms preprocess, 17.4ms inference, 0.3ms postprocess per image at shape (1, 3, 512, 512)

0: 512x512 (no detections), 16.3ms
Speed: 0.5ms preprocess, 16.3ms inference, 0.3ms postprocess per image at shape (1, 3, 512, 512)

0: 512x512 (no detections), 15.9ms
Speed: 0.6ms preprocess, 15.9ms inference, 0.2ms postprocess per image at shape (1, 3, 512, 512)

0: 512x512 (no detections), 16.4ms
Speed: 0.6ms preprocess, 16.4ms i